# Tutorial 3: Moments

`Moments` represent 2D image moments of arbitrary order — the summary statistics
that characterize a spot diagram's size, shape, and asymmetry.

StarSharp uses a generic class-factory pattern: `Moments[n]` returns the concrete
dataclass for order `n`.  Specialized classes `Moments2`, `Moments3`, `Moments4`
are pre-defined with extra properties.

## Key concepts

- **Order 2** — size and ellipticity: `xx`, `xy`, `yy` (trace `T`, ellipticities `e1`, `e2`)
- **Order 3** — skewness/trefoil: `xxx`, `xxy`, `xyy`, `yyy`
- **Order 4** — kurtosis: `xxxx`, `xxxy`, `xxyy`, `xyyy`, `yyyy`
- **Spin decomposition** — moments can be decomposed into complex spin-*m* components

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import Angle

from StarSharp import FieldCoords, Spots, Moments, Moments2, Moments3, Moments4

%matplotlib inline

## 1. The `Moments[n]` class factory

Use `Moments[n]` to get the concrete class for order `n`.  Each class has
fields named by symmetric combinations of `x` and `y`.

In [ ]:
# Get the concrete class for each order
print(f"Moments[2] = {Moments[2].__name__}")
print(f"Moments[3] = {Moments[3].__name__}")
print(f"Moments[4] = {Moments[4].__name__}")
print(f"Moments[5] = {Moments[5].__name__}")  # arbitrary orders work too

In [ ]:
# The pre-imported Moments2, Moments3, Moments4 are the same objects
print(f"Moments[2] is Moments2: {Moments[2] is Moments2}")
print(f"Moments[3] is Moments3: {Moments[3] is Moments3}")
print(f"Moments[4] is Moments4: {Moments[4] is Moments4}")

## 2. Constructing Moments directly

Each moment field is an `astropy.units.Quantity`.  The `frame` parameter records
the coordinate frame (default: `'ocs'`).

In [ ]:
rtp = Angle(30, unit=u.deg)

# Construct Moments2 directly (3 field points)
m2 = Moments2(
    xx=np.array([25.0, 18.0, 30.0]) * u.um**2,
    xy=np.array([2.0, -1.5, 0.5]) * u.um**2,
    yy=np.array([9.0, 12.0, 8.0]) * u.um**2,
    frame="ccs",
    rtp=rtp,
)

print(f"xx = {m2.xx}")
print(f"xy = {m2.xy}")
print(f"yy = {m2.yy}")
print(f"frame = {m2.frame}")

## 3. Moments2 specializations: T, e1, e2

`Moments2` has special properties for common image-quality metrics:

- **T** = `xx + yy` — trace (proportional to spot size squared)
- **e1** = `(xx - yy) / T` — ellipticity component 1
- **e2** = `2 * xy / T` — ellipticity component 2

In [ ]:
print(f"T  = {m2.T}")
print(f"e1 = {m2.e1}")
print(f"e2 = {m2.e2}")

In [ ]:
# Verify the definitions
np.testing.assert_allclose(m2.T, m2.xx + m2.yy)
np.testing.assert_allclose(m2.e1, (m2.xx - m2.yy) / m2.T)
np.testing.assert_allclose(m2.e2, 2 * m2.xy / m2.T)
print("Definitions verified ✓")

## 4. Frame conversions

Moments rotate as tensors.  The `.ocs`, `.ccs`, `.dvcs`, `.edcs` properties
handle the appropriate transformation:
- OCS ↔ CCS: full tensor rotation by ±rtp
- CCS ↔ EDCS: relabel (same values)
- CCS ↔ DVCS: swap x↔y

In [ ]:
m2_ocs = m2.ocs
print(f"CCS: xx={m2.xx[0]:.3f}, xy={m2.xy[0]:.3f}, yy={m2.yy[0]:.3f}")
print(f"OCS: xx={m2_ocs.xx[0]:.3f}, xy={m2_ocs.xy[0]:.3f}, yy={m2_ocs.yy[0]:.3f}")
print()

# Trace is invariant under rotation
print(f"CCS T = {m2.T[0]:.3f}")
print(f"OCS T = {m2_ocs.T[0]:.3f}")
print(f"Trace preserved: {np.allclose(m2.T, m2_ocs.T)}")

In [ ]:
# Roundtrip: CCS → OCS → CCS
m2_rt = m2.ocs.ccs
print(f"xx roundtrip: {np.allclose(m2.xx, m2_rt.xx)}")
print(f"xy roundtrip: {np.allclose(m2.xy, m2_rt.xy)}")
print(f"yy roundtrip: {np.allclose(m2.yy, m2_rt.yy)}")

In [ ]:
# DVCS swaps x↔y, so xx↔yy and xy stays the same
m2_dvcs = m2.dvcs
print(f"CCS  xx={m2.xx[0]:.3f}, yy={m2.yy[0]:.3f}")
print(f"DVCS xx={m2_dvcs.xx[0]:.3f}, yy={m2_dvcs.yy[0]:.3f}")
print(f"DVCS.xx == CCS.yy: {np.allclose(m2_dvcs.xx, m2.yy)}")

### Higher-order frame conversion

In [ ]:
# Order 3 moments: rotation mixes all four components
m3 = Moments3(
    xxx=np.array([5.0]) * u.um**3,
    xxy=np.array([1.0]) * u.um**3,
    xyy=np.array([-2.0]) * u.um**3,
    yyy=np.array([3.0]) * u.um**3,
    frame="ocs",
    rtp=rtp,
)

m3_ccs = m3.ccs
print("OCS → CCS for order-3 moments:")
print(f"  xxx: {m3.xxx[0]:.3f} → {m3_ccs.xxx[0]:.3f}")
print(f"  xxy: {m3.xxy[0]:.3f} → {m3_ccs.xxy[0]:.3f}")
print(f"  xyy: {m3.xyy[0]:.3f} → {m3_ccs.xyy[0]:.3f}")
print(f"  yyy: {m3.yyy[0]:.3f} → {m3_ccs.yyy[0]:.3f}")

## 5. Bridge: Spots → Moments

Frame rotation and moment computation commute: you can rotate the spots first
then compute moments, or compute moments first then rotate — the results match.

In [ ]:
# Build some spots
rng = np.random.default_rng(42)
n_field, n_ray = 4, 500

field = FieldCoords(
    x=rng.uniform(-1, 1, n_field) * u.deg,
    y=rng.uniform(-1, 1, n_field) * u.deg,
    frame="ocs", rtp=rtp,
)

spots_ccs = Spots(
    dx=rng.normal(0, 5, (n_field, n_ray)) * u.um,
    dy=rng.normal(0, 3, (n_field, n_ray)) * u.um,
    vignetted=np.zeros((n_field, n_ray), dtype=bool),
    field=field,
    frame="ccs",
    rtp=rtp,
)

In [ ]:
# Route A: rotate spots to OCS, then compute moments
m2_route_a = spots_ccs.ocs.moments(order=2)

# Route B: compute moments in CCS, then rotate to OCS
m2_route_b = spots_ccs.moments(order=2).ocs

print(f"Route A frame: {m2_route_a.frame}")
print(f"Route B frame: {m2_route_b.frame}")
print(f"xx match: {np.allclose(m2_route_a.xx, m2_route_b.xx)}")
print(f"xy match: {np.allclose(m2_route_a.xy, m2_route_b.xy)}")
print(f"yy match: {np.allclose(m2_route_a.yy, m2_route_b.yy)}")
print(f"Max error: {np.max(np.abs(m2_route_a.xx - m2_route_b.xx)):.2e}")

## 6. Spin decomposition

Moments can be decomposed into **spin-*m* components** — a natural language for
describing the rotational symmetry of each contribution.

For order *N*, spin *m* must satisfy 0 ≤ *m* ≤ *N* and have the same parity as *N*.

The complex spin-*m* moment is $M_m = \langle z^p \bar{z}^q \rangle$ where
$p = (N+m)/2$, $q = (N-m)/2$, and $z = x + iy$.

In [ ]:
m2_demo = spots_ccs.moments(order=2)

# Order 2: spin 0 (size) and spin 2 (ellipticity)
spin0 = m2_demo.spin_complex(0)
spin2 = m2_demo.spin_complex(2)

print(f"Spin-0 (trace/size): {spin0[0]:.3f}")
print(f"  Should equal T:    {m2_demo.T[0]:.3f}")
print()
print(f"Spin-2 (ellipticity): {spin2[0]:.3f}")
print(f"  Real part  (xx-yy): {(m2_demo.xx[0] - m2_demo.yy[0]):.3f}")
print(f"  Imag part  (2·xy):  {(2*m2_demo.xy[0]):.3f}")

In [ ]:
# spin_pair returns (cos, sin) components: (Re, Im) of the complex moment
cos2, sin2 = m2_demo.spin_pair(2)
print(f"spin_pair(2): cos={cos2[0]:.3f}, sin={sin2[0]:.3f}")

# spin_cos and spin_sin for individual access
print(f"spin_cos(2): {m2_demo.spin_cos(2)[0]:.3f}")
print(f"spin_sin(2): {m2_demo.spin_sin(2)[0]:.3f}")

# By convention, negative m is the sin term and positive m is the cos term
print(f"spin(2): {m2_demo.spin(2)[0]:.3f}")
print(f"spin(-2): {m2_demo.spin(-2)[0]:.3f}")

In [ ]:
# Order 3: spins 1 and 3
m3_demo = spots_ccs.moments(order=3)
spin1 = m3_demo.spin_complex(1)  # coma-like
spin3 = m3_demo.spin_complex(3)  # trefoil
print(f"Spin-1 (coma):    {spin1[0]:.3f}")
print(f"Spin-3 (trefoil): {spin3[0]:.3f}")

## 7. Summary

| Property / Method | Description |
|---|---|
| `Moments[n]` | Class factory for order-*n* moments |
| `xx`, `xy`, `yy`, ... | Moment components (Quantity) |
| `.T`, `.e1`, `.e2` | Moments2-only: trace, ellipticities |
| `.ocs`, `.ccs`, `.dvcs`, `.edcs` | Frame conversions (tensor rotation) |
| `.spin_complex(m)` | Complex spin-*m* moment |
| `.spin_pair(m)` | (cos, sin) spin-*m* components |
| `Spots.compute_moments(order)` | Bridge from Spots → Moments |

**Next:** [04_Zernikes.ipynb](04_Zernikes.ipynb) — wavefront coefficients and
their connection to double Zernikes.